# Budowa modelu generującego bajkę w oparciu o 13 bajek braci Grimm. 
## Źródło bajek: https://wolnelektury.pl/katalog/lektury/basnie-bajki-bajeczki/

In [1]:
def laczenie_bajek(lista_plikow):
    teksty_bajek = ""
    for p in lista_plikow:
        with open(p, "r", encoding = "utf-8") as f:
            for line in f:
                line = line.rstrip()
                teksty_bajek += line + " "
    return teksty_bajek

In [2]:
lista_bajek = ['grimm-braciszek-i-siostrzyczka.txt', 'czerwony-kapturek.txt', 'jas-i-malgosia.txt', 'jednooczka-dwuoczka-trzyoczka.txt', 'kopciuszek.txt', 'krol-drozdobrody.txt',
               'krol-zab.txt', 'o-czterech-muzykantach-z-bremy.txt', 'paluszek.txt', 'roszpunka.txt',
               'sniezka.txt', 'stoliczku-nakryj-sie.txt', 'titelitury.txt']

In [3]:
caly_tekst = laczenie_bajek(lista_bajek)

In [4]:
import spacy
nlp = spacy.load("pl_core_news_sm")

In [5]:
caly_tekst = nlp(caly_tekst)

In [6]:
sentences = [sent.text for sent in caly_tekst.sents]

In [7]:
#Wyznaczenie średniej długości zdania w wykorzystanych tekstach (średnia ilość slow w zdaniu)
list_len_sentence = []
for elem in sentences:
    elem = nlp(elem)
    words = [token.text for token in elem]
    len_sentence = len(words)
    list_len_sentence.append(len_sentence)

In [8]:
#Średnia ilość i odchylenie standardowe ilości słów w zdaniu:
import statistics
mean_num_of_words = statistics.mean(list_len_sentence)
st_dev = statistics.stdev(list_len_sentence)

mean_num_of_words = round(mean_num_of_words)
st_dev = round(st_dev)

print(mean_num_of_words)
print(st_dev)

19
11


In [9]:
#Tworzenie n-gramów

In [10]:
import string
words = [token.text for token in caly_tekst]
words = [x.lower() for x in words if x not in string.punctuation and x != " "]
print(words[:100])

['jacob', 'i', 'wilhelm', 'grimm', 'braciszek', 'i', 'siostrzyczka', 'tłum', 'marceli', 'tarnowski', 'isbn', '978', '83', '288', '2267', '2', '  ', 'braciszek', 'ujął', 'siostrzyczkę', 'za', 'rękę', 'i', 'rzekł', '—', 'od', 'śmierci', 'naszej', 'mateczki', 'nie', 'zaznaliśmy', 'chwili', 'radości', 'macocha', 'bije', 'nas', 'i', 'kopie', 'kiedy', 'się', 'do', 'niej', 'zbliżymy', 'a', 'za', 'cały', 'pokarm', 'służą', 'nam', 'twarde', 'skórki', 'od', 'chleba', 'psu', 'pod', 'stołem', 'lepiej', 'się', 'dzieje', 'niż', 'nam', 'spada', 'mu', 'przynajmniej', 'czasem', 'smaczny', 'kąsek', 'o', 'boże', 'gdyby', 'o', 'tym', 'nasza', 'mateczka', 'wiedziała', 'chodź', 'ruszymy', 'razem', 'w', 'daleki', 'świat', 'poszły', 'dzieci', 'przed', 'siebie', 'i', 'szły', 'cały', 'dzień', 'przez', 'łąki', 'i', 'pola', 'a', 'gdy', 'deszcz', 'począł', 'padać', 'rzekła', 'siostrzyczka']


In [11]:
from nltk import ngrams

N3_bajki = list(ngrams(words,3))
print(N3_bajki[:5])

N2_bajki = list(ngrams(words,2))
print(N2_bajki[:5])

[('jacob', 'i', 'wilhelm'), ('i', 'wilhelm', 'grimm'), ('wilhelm', 'grimm', 'braciszek'), ('grimm', 'braciszek', 'i'), ('braciszek', 'i', 'siostrzyczka')]
[('jacob', 'i'), ('i', 'wilhelm'), ('wilhelm', 'grimm'), ('grimm', 'braciszek'), ('braciszek', 'i')]


In [12]:
#Sprawdzanie statystyk dotyczących wszystkich bajek
all_words_in_bajki = len(words)
all_sentence_in_bajki = len(sentences)
print(f'Ilość słów w wszystkich bajkach {all_words_in_bajki}. Ilość zdań we wszystkich bajkach {all_sentence_in_bajki}')

Ilość słów w wszystkich bajkach 22747. Ilość zdań we wszystkich bajkach 1491


In [13]:
#Tworzenie słowników na podstawie otrzymanych n-gramów (słowniki typu (słowo1, słowo 2): słowo 3)
N3_bajki_dict = {}

for (word_1, word_2, word_3) in N3_bajki:
    if (word_1, word_2) not in N3_bajki_dict:
        N3_bajki_dict[(word_1, word_2)] = []
    N3_bajki_dict[(word_1, word_2)].append(word_3)

N2_bajki_dict = {}
for (word_1, word_2) in N2_bajki:
    if (word_1) not in N2_bajki_dict:
        N2_bajki_dict[(word_1)] = []
    N2_bajki_dict[(word_1)].append(word_2)

In [14]:
#Uwzględnienie prawdopodobieństwa przejść w n-gramach
def word_probability(dictionary):
    prob_dict = {}
    for key, word_list in dictionary.items():
        count = {}
        for word in word_list:
            count[word] = count.get(word, 0) + 1
        all_words = len(word_list)
        
        prob_dict[key] = {}
        for word, amount in count.items():
            prob_dict[key][word] = amount/all_words
    return prob_dict

In [15]:
#tworzenie słowników z prawdopodobieńśtwami dla 3-gramów i 2-gramów
N3_prob = word_probability(N3_bajki_dict)
N2_prob = word_probability(N2_bajki_dict)

In [16]:
print(N2_prob.get("jest"))

{'zraniony': 0.011904761904761904, 'na': 0.4166666666666667, 'w': 0.16666666666666666, 'chora': 0.011904761904761904, 'przecież': 0.011904761904761904, 'jeszcze': 0.011904761904761904, 'dość': 0.011904761904761904, 'pasiona': 0.011904761904761904, 'ich': 0.011904761904761904, 'moją': 0.011904761904761904, 'z': 0.011904761904761904, 'ta': 0.011904761904761904, 'tego': 0.011904761904761904, 'miły': 0.011904761904761904, 'tak': 0.03571428571428571, 'chodzić': 0.011904761904761904, 'wciągnąć': 0.011904761904761904, 'przy': 0.011904761904761904, 'tysiąc': 0.023809523809523808, 'piękne': 0.011904761904761904, 'znów': 0.011904761904761904, 'za': 0.011904761904761904, 'stolik': 0.011904761904761904, 'osioł': 0.011904761904761904, 'to': 0.03571428571428571, 'niczym': 0.011904761904761904, 'pewnie': 0.011904761904761904, 'kudłaczu': 0.011904761904761904, 'moje': 0.03571428571428571, 'titelitury': 0.011904761904761904}


In [17]:
print(N3_prob.get(("pewnego", "razu")))

{'król': 0.06666666666666667, 'podarowała': 0.06666666666666667, 'rzekła': 0.06666666666666667, 'siadła': 0.06666666666666667, 'stały': 0.06666666666666667, 'przyszły': 0.06666666666666667, 'kazał': 0.06666666666666667, 'miało': 0.06666666666666667, 'że': 0.13333333333333333, 'stał': 0.06666666666666667, '—': 0.06666666666666667, 'do': 0.06666666666666667, 'podczas': 0.06666666666666667, 'królowa': 0.06666666666666667}


In [18]:
#losowanie słów z prawdopodobieństwem
import random
def draw_with_prob(prob_dict):
    words = list(prob_dict.keys())
    weights = list(prob_dict.values())
    return random.choices(words, weights = weights)[0]

In [19]:
#Generowanie zdania
import random
def make_sentence(N3_prob, N2_prob, words, mean_num_of_words, st_dev, sentence_start = None):
    #losowa ilość słów w zdaniu
    random_len_sent = round(random.gauss(mean_num_of_words, st_dev))
    random_len_sent = max(3, random_len_sent) #zabezpieczenie przed powstaniem zdania krótszego niż 3 słowa.
    
    #Losowy start lub możliwość wybrania dwoch pierwszych słów zdania
    if sentence_start == None:
        klucze = list(N3_prob.keys())
        start = random.choice(klucze)
        sentence = list(start)
    else:
        sentence = list(sentence_start)

    for i in range (random_len_sent - 2):
        contex_2_words = (sentence[-2], sentence[-1])
        contex_1_word = (sentence[-1])

        if contex_2_words in N3_prob:
            next_word = draw_with_prob(N3_prob[contex_2_words])
        elif contex_1_word in N2_prob:
            next_word = draw_with_prob(N2_prob[contex_1_word])
        else:
            next_word = random.choice(words)

        sentence.append(next_word)

    #Dodawanie dużej litery na początku zdania i kropki na końcu:
    sentence = " ".join(sentence) + "."
    sentence = sentence.capitalize()
    #Użycie spacy do wygenerowania pos_tagów
    sent_for_tagg = nlp(sentence)
    tagged = [(token.text, token.pos_) for token in sent_for_tagg]

    return sentence, tagged

In [20]:
zdanie_probne = make_sentence(N3_prob, N2_prob, words, mean_num_of_words, st_dev)
print(zdanie_probne)

('Zobaczył w nim fałszywa królowa kiedy nadeszła oznaczona pora matka zaprzęgła konie do wozu i wsadziła paluszka do ucha — sprzedajcie nam tego malca będzie mu u mnie —.', [('Zobaczył', 'VERB'), ('w', 'ADP'), ('nim', 'PRON'), ('fałszywa', 'ADJ'), ('królowa', 'NOUN'), ('kiedy', 'ADV'), ('nadeszła', 'VERB'), ('oznaczona', 'ADJ'), ('pora', 'ADJ'), ('matka', 'NOUN'), ('zaprzęgła', 'VERB'), ('konie', 'NOUN'), ('do', 'ADP'), ('wozu', 'NOUN'), ('i', 'CCONJ'), ('wsadziła', 'VERB'), ('paluszka', 'NOUN'), ('do', 'ADP'), ('ucha', 'NOUN'), ('—', 'PUNCT'), ('sprzedajcie', 'VERB'), ('nam', 'PRON'), ('tego', 'DET'), ('malca', 'NOUN'), ('będzie', 'AUX'), ('mu', 'PRON'), ('u', 'ADP'), ('mnie', 'PRON'), ('—', 'PUNCT'), ('.', 'PUNCT')])


In [21]:
zdanie_probne_z_podanym_startem = make_sentence(N3_prob, N2_prob, words, mean_num_of_words, st_dev, sentence_start = ("pewnego", "razu"))
print(zdanie_probne_z_podanym_startem)

('Pewnego razu król urządził wielkie polowanie po całym świecie możecie sobie wyobrazić jak się ściskały za szyje i całowały a ponieważ zła czarownica już nie ożyjesz — rzekła siostrzyczka.', [('Pewnego', 'DET'), ('razu', 'NOUN'), ('król', 'NOUN'), ('urządził', 'VERB'), ('wielkie', 'ADJ'), ('polowanie', 'NOUN'), ('po', 'ADP'), ('całym', 'ADJ'), ('świecie', 'NOUN'), ('możecie', 'VERB'), ('sobie', 'PRON'), ('wyobrazić', 'VERB'), ('jak', 'SCONJ'), ('się', 'PRON'), ('ściskały', 'VERB'), ('za', 'ADP'), ('szyje', 'NOUN'), ('i', 'CCONJ'), ('całowały', 'VERB'), ('a', 'CCONJ'), ('ponieważ', 'SCONJ'), ('zła', 'ADJ'), ('czarownica', 'NOUN'), ('już', 'PART'), ('nie', 'PART'), ('ożyjesz', 'VERB'), ('—', 'PUNCT'), ('rzekła', 'VERB'), ('siostrzyczka', 'NOUN'), ('.', 'PUNCT')])


In [24]:
sample_text = []
tagged_text = []
for i in range(20):
    sentence, tagged = make_sentence(N3_prob, N2_prob, words, mean_num_of_words, st_dev)
    sample_text.append(sentence)
    tagged_text.append(tagged)
final_text = " ".join(sample_text)
print(final_text)
print(tagged_text)

Żaba stuknęła we drzwi wejściowe i. Jasiu wysuń palec chcę zobaczyć czy jesteś już dość tłusty ale jaś nie utyje chyba nigdy więc muszę go zdobyć koniecznie gdy nadeszła pora snu gość wyciągnął się na ławie i podłożył worek. Świat kiedy mu. Czas daremnie pragnęli dziecka wreszcie żona powzięła nadzieję że pan bóg płacze razem z tobą ale nie mogli znaleźć wyjścia z lasu. Świeża i zielona że kobieta zapragnęła jej skosztować i myśl ta nie odstępowała. Zawołał stamtąd — do widzenia moi panowie ruszajcie do domu — i byli nim tak ucieszeni że nie słyszysz. Czekać — hejże małgosiu — zawołała a królewicz uradowany odpowiedział. Nie spadła do. Cudnie wyglądała na białym śniegu królowa pomyślała sobie „ nie będzie się już zupełnie ściemniło przyszli. Srebrem jakaś nieziemska postać stanęła pośrodku kuchni była to żaba która dowlókłszy się do studni królewna goniła ją oczyma ale kula zniknęła bo studnia była głęboka. Dobry bóg nie opuści po długim szukaniu złodzieje znaleźli go wreszcie i podnie

In [23]:
sample_text_with_beggining = []
tagged_text = []
for i in range(20):
    if i == 0:
        sentence, tagged = make_sentence(N3_prob, N2_prob, words, mean_num_of_words, st_dev, sentence_start = ("pewnego", "razu"))
        sample_text_with_beggining.append(sentence)
        tagged_text.append(tagged)
    else:
        sentence, tagged = make_sentence(N3_prob, N2_prob, words, mean_num_of_words, st_dev)
        sample_text_with_beggining.append(sentence)
        tagged_text.append(tagged)
final_text = " ".join(sample_text_with_beggining)
print(final_text)
print(tagged_text)

Pewnego razu stał za drzewem. Przystanął więc i począł okładać złego karczmarza karczmarz krzyczał i błagał aż siostrzyczka pozwoliła mu. Biegła tak po kamieniach braciszek chciał się do ogrodu narwał szybko roszpunki i zaniósł żonie kobieta przyrządziła sobie z pewnego króla który stał na samym przodzie i miał podbródek nieco zbyt wydatny — ach. Teraz stoi za drzwiami i chce się ze mną do zamku kiedy dziewczyna przyszła do pokoju proboszcza i podam wam stamtąd wszystko czego chcecie wyciągnijcie tylko ręce usłyszała to kucharka i poczęła nasłuchiwać ale złodzieje uciekli już kawałek. Wielkie uszy — abym cię mogła poznać zapukaj trzy razy i zawołał    roszpunko dziewczę moje spuśćże mi włosy swoje wnet opadły włosy i rzuciła. Wyciągnął stamtąd malca — ach to ty stara wodochlapko — rzekła wieśniaczka — odchodzę już z moimi jabłkami tylko jedno oko robiła jej wyrzuty mówiąc. Znowu tymczasem siostrzyczka przeraziła się bardzo —. Musiała my wyjeżdżamy a jeżeli za powrotem nie ma ani lampy 